In [3]:
# ============================================================
# CELL 1 - INSTALLATION
# ============================================================

print("📦 Libraries install ho rahi hain...")

!pip uninstall -y google-generativeai
!pip install -q google-genai gradio pillow pandas numpy scikit-learn matplotlib opencv-python-headless
!apt-get install -q tesseract-ocr 2>/dev/null

print("✅ Sab install ho gaya! Ab Cell 2 chalao.")

📦 Libraries install ho rahi hain...
Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
✅ Sab install ho gaya! Ab Cell 2 chalao.


In [ ]:
# ============================================================
# CELL 2 - API KEY + ALL IMPORTS
# ============================================================

# ── Imports ──────────────────────────────────────────────
import os, re, json, warnings, io
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2

# ✅ NAYA IMPORT YAHAN HAI
from google import genai
from google.genai import types

from PIL import Image, ImageEnhance, ImageFilter
from sklearn.ensemble import RandomForestRegressor
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
print("✅ Imports done!")

# ── API KEY SETUP ────────────────────────────────────────
# ⚠️ SECURITY: Never commit real API keys!
# Setup: Copy .env.example to .env and add your real key there
try:
    from dotenv import load_dotenv
    load_dotenv()
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', 'your-api-key-here')
except:
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', 'your-api-key-here')

if GEMINI_API_KEY == 'your-api-key-here':
    print("⚠️ WARNING: API key not set! Set GEMINI_API_KEY environment variable")

# ✅ NAYA CLIENT SETUP
client = genai.Client(api_key=GEMINI_API_KEY)

# ── Test karo ki key sahi hai ──
try:
    test = client.models.generate_content(
        model='gemini-2.5-flash',
        contents="Say OK in one word"
    )
    print(f"✅ Gemini connected! Response: {test.text.strip()}")
except Exception as e:
    print(f"❌ Gemini Error: {e}")
    print("   API key check karo!")

print("\n✅ Cell 2 done! Ab Cell 3 chalao.")

✅ Imports done!
✅ Gemini connected! Response: Okay.

✅ Cell 2 done! Ab Cell 3 chalao.


In [20]:
# ============================================================
# CELL 3A - SHOP DATA STORE
# ============================================================

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

class ShopDataStore:
    def __init__(self):
        self.inventory = {}
        self.sales_history = []
        self._initialize_default_inventory()

    def _initialize_default_inventory(self):
        self.inventory = {
            'Pencil':         {'stock': 45,  'avg_daily_sale': 15, 'reorder': 50,  'price': 5},
            'Pen Blue':       {'stock': 60,  'avg_daily_sale': 20, 'reorder': 60,  'price': 10},
            'Pen Black':      {'stock': 30,  'avg_daily_sale': 12, 'reorder': 40,  'price': 10},
            'Eraser':         {'stock': 80,  'avg_daily_sale': 10, 'reorder': 30,  'price': 3},
            'Sharpener':      {'stock': 25,  'avg_daily_sale': 8,  'reorder': 25,  'price': 5},
            'Notebook A4':    {'stock': 20,  'avg_daily_sale': 6,  'reorder': 20,  'price': 40},
            'Notebook A5':    {'stock': 15,  'avg_daily_sale': 8,  'reorder': 25,  'price': 25},
            'Stapler':        {'stock': 10,  'avg_daily_sale': 1,  'reorder': 5,   'price': 150},
            'Staple Pins':    {'stock': 12,  'avg_daily_sale': 4,  'reorder': 15,  'price': 20},
            'Ruler':          {'stock': 35,  'avg_daily_sale': 5,  'reorder': 15,  'price': 15},
            'Geometry Box':   {'stock': 8,   'avg_daily_sale': 3,  'reorder': 10,  'price': 120},
            'Colour Pencils': {'stock': 18,  'avg_daily_sale': 4,  'reorder': 12,  'price': 80},
            'Marker':         {'stock': 22,  'avg_daily_sale': 6,  'reorder': 18,  'price': 30},
            'Highlighter':    {'stock': 16,  'avg_daily_sale': 5,  'reorder': 15,  'price': 25},
            'Glue Stick':     {'stock': 20,  'avg_daily_sale': 5,  'reorder': 15,  'price': 20},
            'Scissors':       {'stock': 14,  'avg_daily_sale': 2,  'reorder': 8,   'price': 50},
            'Drawing Book':   {'stock': 12,  'avg_daily_sale': 3,  'reorder': 10,  'price': 60},
            'Calculator':     {'stock': 5,   'avg_daily_sale': 1,  'reorder': 3,   'price': 200},
            'File Folder':    {'stock': 25,  'avg_daily_sale': 4,  'reorder': 12,  'price': 30},
            'Sticky Notes':   {'stock': 10,  'avg_daily_sale': 3,  'reorder': 10,  'price': 35},
        }
        self._generate_initial_history()

    def _generate_initial_history(self):
        np.random.seed(42)
        start = datetime.now() - timedelta(days=30)
        for day in range(30):
            date = start + timedelta(days=day)
            for item, props in self.inventory.items():
                noise   = max(0.2, np.random.normal(1.0, 0.25))
                weekend = 1.3 if date.weekday() >= 5 else 1.0
                sold    = max(0, int(props['avg_daily_sale'] * noise * weekend))
                self.sales_history.append({
                    'date':        date.strftime('%Y-%m-%d'),
                    'item':        item,
                    'units_sold':  sold,
                    'day_of_week': date.weekday(),
                    'month':       date.month,
                    'is_weekend':  1 if date.weekday() >= 5 else 0,
                })

    def add_sale(self, item, quantity, source='manual'):
        if item in self.inventory:
            self.inventory[item]['stock'] = max(
                0, self.inventory[item]['stock'] - quantity
            )
            date = datetime.now()
            self.sales_history.append({
                'date':        date.strftime('%Y-%m-%d'),
                'item':        item,
                'units_sold':  quantity,
                'day_of_week': date.weekday(),
                'month':       date.month,
                'is_weekend':  1 if date.weekday() >= 5 else 0,
            })
            recent = [s['units_sold'] for s in self.sales_history
                      if s['item'] == item][-30:]
            self.inventory[item]['avg_daily_sale'] = round(np.mean(recent), 1)
            return True
        return False

    def add_stock(self, item, quantity):
        if item in self.inventory:
            self.inventory[item]['stock'] += quantity
            return True
        return False

    def add_new_item(self, name, stock, avg_sale, reorder, price):
        self.inventory[name] = {
            'stock':          int(stock),
            'avg_daily_sale': float(avg_sale),
            'reorder':        int(reorder),
            'price':          float(price),
        }
        return True

    def get_inventory_df(self):
        rows = []
        for item, p in self.inventory.items():
            days = (p['stock'] / p['avg_daily_sale']
                    if p['avg_daily_sale'] > 0 else 999)
            if   days <= 3:  risk = '🔴 CRITICAL'
            elif days <= 7:  risk = '🟠 HIGH'
            elif days <= 14: risk = '🟡 MEDIUM'
            else:            risk = '🟢 SAFE'
            rows.append({
                'Item':           item,
                'Stock':          p['stock'],
                'Avg Daily Sale': p['avg_daily_sale'],
                'Days Left':      round(days, 1),
                'Reorder Point':  p['reorder'],
                'Price (₹)':      p['price'],
                'Risk':           risk,
            })
        return pd.DataFrame(rows).sort_values('Days Left')


# ── Global instance ──────────────────────────────────────
store = ShopDataStore()

print("✅ ShopDataStore ready!")
print(f"   📦 Items loaded  : {len(store.inventory)}")
print(f"   📅 History rows  : {len(store.sales_history)}")
print("\n✅ Cell 3A done! Ab Cell 3B chalao.")

✅ ShopDataStore ready!
   📦 Items loaded  : 20
   📅 History rows  : 600

✅ Cell 3A done! Ab Cell 3B chalao.


In [22]:
# ============================================================
# CELL 3B - GEMINI OCR ENGINE (Gemini 2.5 Flash)
# ============================================================

import re, json
import pandas as pd
from PIL import Image, ImageEnhance
from google.genai import types

class GeminiOCR:

    def __init__(self, inventory_keys):
        self.known_items = list(inventory_keys)
        self.model_name = 'gemini-2.5-flash' # ✅ Direct 2.5 Flash set kar diya

    def _enhance(self, image):
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        w, h = image.size
        if w < 400 or h < 200:
            image = image.resize((w*2, h*2), Image.LANCZOS)
        image = ImageEnhance.Contrast(image).enhance(1.8)
        image = ImageEnhance.Sharpness(image).enhance(2.0)
        return image

    def _match(self, extracted):
        if not extracted:
            return 'Unknown'
        ext = extracted.lower()
        for k in self.known_items:
            if k.lower() == ext: return k
        for k in self.known_items:
            if k.lower() in ext or ext in k.lower(): return k
        kw = {
            'pencil': 'Pencil', 'pen': 'Pen Blue', 'eraser': 'Eraser',
            'rubber': 'Eraser', 'sharp': 'Sharpener', 'notebook': 'Notebook A4',
            'copy': 'Notebook A5', 'ruler': 'Ruler', 'scale': 'Ruler',
            'stapl': 'Stapler', 'geomet': 'Geometry Box', 'colour': 'Colour Pencils',
            'color': 'Colour Pencils', 'marker': 'Marker', 'highlight': 'Highlighter',
            'glue': 'Glue Stick', 'scissor': 'Scissors', 'calculat': 'Calculator',
            'folder': 'File Folder', 'sticky': 'Sticky Notes', 'drawing': 'Drawing Book',
        }
        for k, v in kw.items():
            if k in ext: return v
        return extracted.title()

    def _parse_response(self, raw_text):
        try:
            text = raw_text.strip()
            if text.startswith('```json'):
                text = text[7:]
            elif text.startswith('```'):
                text = text[3:]

            if text.endswith('```'):
                text = text[:-3]

            text = text.strip()

            m = re.search(r'\{.*\}', text, re.DOTALL)
            if m:
                return json.loads(m.group())
            return json.loads(text)
        except Exception as e:
            print(f"JSON Parse Error: {e}")
            return None

    def from_image(self, image):
        if image is None:
            return ("❌ Image upload karo!", pd.DataFrame(columns=['Item','Quantity','Unit','Brand','Confidence','Original Text']))
        try:
            enhanced = self._enhance(image)
            prompt = f"""
You are an expert OCR system for an Indian stationary shop.
Read ALL text from the image carefully — including handwritten text.
Known items in shop: {', '.join(self.known_items)}
Rules:
1. Read every word carefully, even if handwritten or unclear
2. Map brand + item to item type only:
   • Apsara/Natraj Pencil  → item: Pencil,    brand: Apsara/Natraj
   • Cello/Reynolds Pen    → item: Pen Blue,  brand: Cello/Reynolds
3. Parse quantity units correctly: "1 BOX" → unit:box | "5 pcs" → unit:pcs
4. Default quantity = 1 if not written

Return ONLY this JSON:
{{
  "raw_text": "<exact text seen in image>",
  "image_type": "bill/invoice",
  "items": [
    {{
      "item": "<matched known item>",
      "quantity": 1,
      "unit": "pcs",
      "brand": "<brand>",
      "confidence": "high",
      "original_text": "<text>"
    }}
  ]
}}"""

            resp = client.models.generate_content(
                model=self.model_name,
                contents=[prompt, enhanced],
                config=types.GenerateContentConfig(temperature=0.1)
            )

            data = self._parse_response(resp.text.strip())
            if data is None:
                return (f"⚠️ Parse error:\n{resp.text[:400]}", pd.DataFrame(columns=['Item','Quantity','Unit','Brand','Confidence','Original Text']))

            items = data.get('items', [])
            raw   = data.get('raw_text', 'N/A')
            itype = data.get('image_type', 'unknown')

            if not items:
                return (f"⚠️ Koi item nahi mila.\nImage type : {itype}\nRaw text : {raw}", pd.DataFrame(columns=['Item','Quantity','Unit','Brand','Confidence','Original Text']))

            status = f"✅ {len(items)} items mile!\n{'─'*40}\n🖼️ Type : {itype}\n📝 Text : {raw}\n{'─'*40}\n✏️ Neeche table edit karo"

            rows = [{'Item': self._match(i.get('item', '')), 'Quantity': int(i.get('quantity', 1)), 'Unit': i.get('unit', 'pcs'), 'Brand': i.get('brand', '-'), 'Confidence': i.get('confidence', 'medium'), 'Original Text': i.get('original_text', '-')} for i in items]
            return status, pd.DataFrame(rows)

        except Exception as e:
            return (f"❌ Error: {e}", pd.DataFrame(columns=['Item','Quantity','Unit','Brand','Confidence','Original Text']))

    def from_text(self, text):
        if not text.strip():
            return ("❌ Text daalo!", pd.DataFrame(columns=['Item','Quantity','Unit','Brand','Confidence','Original Text']))
        try:
            prompt = f"""
Extract stationary items from this text for an Indian shop.
Text: "{text}"
Known items: {', '.join(self.known_items)}
Return ONLY JSON:
{{ "items": [ {{"item": "<item>", "quantity": 1, "unit": "pcs", "brand": "<brand>", "confidence": "high"}} ] }}"""

            resp = client.models.generate_content(
                model=self.model_name,
                contents=prompt
            )

            data = self._parse_response(resp.text.strip())
            if data is None:
                return ("⚠️ Parse error!", pd.DataFrame(columns=['Item','Quantity','Unit','Brand','Confidence','Original Text']))

            items = data.get('items', [])
            rows  = [{'Item': self._match(i.get('item', '')), 'Quantity': int(i.get('quantity', 1)), 'Unit': i.get('unit', 'pcs'), 'Brand': i.get('brand', '-'), 'Confidence': i.get('confidence', 'medium'), 'Original Text': text[:60]} for i in items]
            return f"✅ {len(rows)} items mile!", pd.DataFrame(rows)

        except Exception as e:
            return (f"❌ Error: {e}", pd.DataFrame(columns=['Item','Quantity','Unit','Brand','Confidence','Original Text']))

# ── Global instance ──────────────────────────────────────
ocr_engine = GeminiOCR(store.inventory.keys())

print("✅ GeminiOCR ready!")
print(f"   🤖 Model in use: {ocr_engine.model_name}")
print(f"   🔍 Known items : {len(ocr_engine.known_items)}")
print("\n✅ Cell 3B done! Ab Cell 3C chalao.")

✅ GeminiOCR ready!
   🤖 Model in use: gemini-2.5-flash
   🔍 Known items : 20

✅ Cell 3B done! Ab Cell 3C chalao.


In [25]:
# ============================================================
# CELL 3C - PREDICTION ENGINE + CHART HELPERS
# ============================================================
import io
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from datetime import datetime, timedelta
from PIL import Image

class PredictionEngine:
    def __init__(self, data_store):
        self.store  = data_store
        self.models = {}

    def train(self):
        df = pd.DataFrame(self.store.sales_history)
        if df.empty: return
        for item in self.store.inventory:
            idf = df[df['item'] == item].reset_index(drop=True)
            if len(idf) < 5: continue
            idf['day_num'] = range(len(idf))
            X = idf[['day_num', 'day_of_week', 'month', 'is_weekend']]
            y = idf['units_sold']
            try:
                m = RandomForestRegressor(n_estimators=30, random_state=42)
                m.fit(X, y)
                self.models[item] = {'model': m, 'last_day': len(idf)}
            except: pass

    def predict_item(self, item_name, days=30):
        if item_name not in self.store.inventory: return None, []
        props = self.store.inventory[item_name]
        remaining = props['stock']
        today = datetime.now()
        preds, stockout = [], None
        info = self.models.get(item_name)

        for i in range(1, days + 1):
            future = today + timedelta(days=i)
            if info:
                feat = pd.DataFrame({
                    'day_num': [info['last_day'] + i],
                    'day_of_week': [future.weekday()],
                    'month': [future.month],
                    'is_weekend': [1 if future.weekday() >= 5 else 0],
                })
                try: sale = max(0, int(info['model'].predict(feat)[0]))
                except: sale = int(props['avg_daily_sale'] * max(0.2, np.random.normal(1, 0.2)))
            else:
                sale = int(props['avg_daily_sale'] * max(0.2, np.random.normal(1, 0.2)))

            remaining -= sale
            preds.append({'day': i, 'date': future.strftime('%d %b'), 'sale': max(0, sale), 'remaining': max(0, remaining)})
            if remaining <= 0 and stockout is None: stockout = future

        return stockout, preds

    def all_predictions(self):
        self.train()
        results = []
        for item, props in self.store.inventory.items():
            stockout, preds = self.predict_item(item)
            days_left = (props['stock'] / props['avg_daily_sale'] if props['avg_daily_sale'] > 0 else 999)
            risk = '🔴 CRITICAL' if days_left <= 3 else '🟠 HIGH' if days_left <= 7 else '🟡 MEDIUM' if days_left <= 14 else '🟢 SAFE'
            results.append({
                'item': item, 'stock': props['stock'], 'avg_sale': round(props['avg_daily_sale'], 1),
                'days_left': round(days_left, 1), 'stockout': (stockout.strftime('%d %b %Y') if stockout else '>30 days'),
                'risk': risk, 'order_qty': int(props['avg_daily_sale'] * 30), 'preds': preds,
            })
        return sorted(results, key=lambda x: x['days_left'])

CLR = {'🔴 CRITICAL': '#ff4757', '🟠 HIGH': '#ff6b35', '🟡 MEDIUM': '#ffd32a', '🟢 SAFE': '#2ed573'}

def _buf_to_pil(fig):
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=120, bbox_inches='tight', facecolor='#0f0f23')
    buf.seek(0)
    plt.close(fig)
    return Image.open(buf)

def _style_ax(ax):
    ax.set_facecolor('#1a1a3e')
    [sp.set_visible(False) for sp in ax.spines.values()]
    ax.tick_params(colors='white', labelsize=8)
    ax.grid(color='#2a2a4a', alpha=0.5)

def make_dashboard(results, store):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.patch.set_facecolor('#0f0f23')

    ax = axes[0, 0]; _style_ax(ax)
    items = [r['item'] for r in results[:15]]; days = [min(r['days_left'], 35) for r in results[:15]]
    cols = [CLR.get(r['risk'], '#aaa') for r in results[:15]]
    bars = ax.barh(items, days, color=cols, edgecolor='#2a2a4a', height=0.7)
    ax.axvline(7, color='#ff4757', ls='--', lw=1.5, alpha=.8, label='7d')
    ax.axvline(14, color='#ffd32a', ls='--', lw=1.5, alpha=.8, label='14d')
    ax.set_title('📅 Stock Kitne Din Ka?', color='white', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, labelcolor='white', facecolor='#2a2a4a')
    for b, v in zip(bars, days): ax.text(b.get_width() + .3, b.get_y() + b.get_height() / 2, f'{v:.0f}d', va='center', color='white', fontsize=7)

    ax = axes[0, 1]; ax.set_facecolor('#1a1a3e')
    rc = {}
    for r in results:
        k = r['risk'].split(' ', 1)[1]
        rc[k] = rc.get(k, 0) + 1
    if rc:
        labels = list(rc.keys())
        pcols = ['#ff4757' if 'CRIT' in l else '#ff6b35' if 'HIGH' in l else '#ffd32a' if 'MED' in l else '#2ed573' for l in labels]
        _, _, autos = ax.pie(list(rc.values()), labels=labels, colors=pcols, autopct='%1.0f%%', startangle=90, textprops={'color': 'white', 'fontsize': 9}, wedgeprops={'edgecolor': '#0f0f23', 'linewidth': 2})
        [a.set_color('white') for a in autos]
    ax.set_title('⚡ Risk Distribution', color='white', fontsize=11, fontweight='bold')

    ax = axes[1, 0]; _style_ax(ax)
    top = sorted(results, key=lambda x: x['avg_sale'], reverse=True)[:10]
    ax.bar(range(len(top)), [r['avg_sale'] for r in top], color='#5352ed', edgecolor='#2a2a4a')
    ax.set_xticks(range(len(top))); ax.set_xticklabels([r['item'] for r in top], rotation=45, ha='right', color='white', fontsize=7)
    ax.set_title('🏆 Top Sellers', color='white', fontsize=11, fontweight='bold')

    ax = axes[1, 1]; _style_ax(ax)
    ri = ([r for r in results if '🔴' in r['risk'] or '🟠' in r['risk']][:8] or results[:8])
    x = np.arange(len(ri))
    ax.bar(x - .18, [store.inventory[r['item']]['stock'] for r in ri], .35, label='Stock', color='#5352ed', alpha=.85)
    ax.bar(x + .18, [store.inventory[r['item']]['reorder'] for r in ri], .35, label='Reorder', color='#ff4757', alpha=.85)
    ax.set_xticks(x); ax.set_xticklabels([r['item'] for r in ri], rotation=45, ha='right', color='white', fontsize=7)
    ax.set_title('📦 Stock vs Reorder', color='white', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, labelcolor='white', facecolor='#2a2a4a')

    plt.suptitle('🏪 Stationary Shop Dashboard', color='white', fontsize=14, fontweight='bold'); plt.tight_layout()
    return _buf_to_pil(fig)

def make_item_chart(item_name, preds, store):
    if not preds: return None
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.patch.set_facecolor('#0f0f23')
    days = [p['day'] for p in preds]; rem = [p['remaining'] for p in preds]; sales = [p['sale'] for p in preds]
    props = store.inventory.get(item_name, {})
    dl = (props.get('stock', 0) / props.get('avg_daily_sale', 1) if props.get('avg_daily_sale', 0) > 0 else 999)
    risk = '🔴 CRITICAL' if dl <= 3 else '🟠 HIGH' if dl <= 7 else '🟡 MEDIUM' if dl <= 14 else '🟢 SAFE'
    col = '#ff4757' if '🔴' in risk else '#5352ed'
    for ax in (ax1, ax2): _style_ax(ax); [sp.set_color('#3a3a5a') for sp in ax.spines.values()]

    ax1.fill_between(days, rem, alpha=.25, color=col); ax1.plot(days, rem, color=col, lw=2.5, label='Remaining')
    reorder = props.get('reorder', 0)
    ax1.axhline(reorder, color='#ffd32a', ls='--', lw=1.5, label=f'Reorder ({reorder})')
    ax1.set_title(f'📉 {item_name}\\n{risk}', color='white', fontsize=10, fontweight='bold')
    ax1.set_xlabel('Days', color='white', fontsize=9); ax1.set_ylabel('Units', color='white', fontsize=9)
    ax1.legend(fontsize=8, labelcolor='white', facecolor='#2a2a4a')

    avg = props.get('avg_daily_sale', 0)
    bc = ['#ff4757' if s > avg * 1.3 else '#2ed573' for s in sales]
    ax2.bar(days, sales, color=bc, edgecolor='#2a2a4a', alpha=.85)
    ax2.axhline(avg, color='#ffd32a', ls='--', lw=1.5, label=f'Avg ({avg})')
    ax2.set_title('📊 Daily Sales Forecast', color='white', fontsize=10, fontweight='bold')
    ax2.set_xlabel('Days', color='white', fontsize=9); ax2.set_ylabel('Units/Day', color='white', fontsize=9)
    ax2.legend(fontsize=8, labelcolor='white', facecolor='#2a2a4a')

    plt.tight_layout()
    return _buf_to_pil(fig)

# ── Global instance ──────────────────────────────────────
pred_engine = PredictionEngine(store)

print("✅ PredictionEngine ready!")
print("\n✅ Cell 3C done! Ab Cell 4 chalao.")

✅ PredictionEngine ready!

✅ Cell 3C done! Ab Cell 4 chalao.


In [26]:
# ============================================================
# CELL 4 - UI HANDLER FUNCTIONS (FIXED & WHATSAPP ADDED)
# ============================================================
import pandas as pd
from datetime import datetime

# ── OCR Handlers ─────────────────────────────────────────
def scan_image(image):
    return ocr_engine.from_image(image)

def scan_text(text):
    return ocr_engine.from_text(text)

# ── Confirm & Update ─────────────────────────────────────
def confirm_update(df, action):
    try:
        # Gradio DataFrame format handling
        if isinstance(df, dict):
            if 'data' in df:
                df = pd.DataFrame(df['data'], columns=df.get('headers', []))
            else:
                df = pd.DataFrame(df)
        elif isinstance(df, list):
            df = pd.DataFrame(df)

        if df is None or df.empty or len(df) == 0:
            return "❌ Koi data nahi! Pehle OCR scan karo ya table mein data dalo.", store.get_inventory_df()

        updated, errors = [], []
        for _, row in df.iterrows():
            item = str(row.get('Item', '')).strip()
            try:
                qty = int(row.get('Quantity', 0))
            except:
                qty = 0

            if not item or item == 'None' or qty <= 0:
                continue

            if "Sale" in action:
                ok = store.add_sale(item, qty, source='ocr')
            else:
                ok = store.add_stock(item, qty)

            (updated if ok else errors).append(
                f"{'✅' if ok else '❌'} {item}: "
                f"{'-' if 'Sale' in action else '+'}{qty}"
            )

        msg = "✅ Update complete!\n" + "\n".join(updated)
        if errors:
            msg += "\n\nErrors:\n" + "\n".join(errors)
        msg += "\n\n📊 Prediction tab mein jaake results dekho!"
        return msg, store.get_inventory_df()

    except Exception as e:
        import traceback
        return f"❌ Error: {str(e)}\n{traceback.format_exc()}", store.get_inventory_df()

# ── Prediction Handler ───────────────────────────────────
def run_prediction():
    try:
        results = pred_engine.all_predictions()

        critical = [r for r in results if '🔴' in r['risk']]
        high     = [r for r in results if '🟠' in r['risk']]
        medium   = [r for r in results if '🟡' in r['risk']]
        safe     = [r for r in results if '🟢' in r['risk']]

        html = f"""
        <div style="display:grid;grid-template-columns:repeat(4,1fr); gap:12px;padding:10px;">
          <div style="background:linear-gradient(135deg,#ff4757,#c0392b); padding:18px;border-radius:12px;text-align:center;">
            <div style="font-size:32px;font-weight:bold;color:white;">{len(critical)}</div>
            <div style="color:#ffcdd2;font-size:12px;">🔴 CRITICAL<br>≤ 3 days</div>
          </div>
          <div style="background:linear-gradient(135deg,#ff6b35,#e55a2b); padding:18px;border-radius:12px;text-align:center;">
            <div style="font-size:32px;font-weight:bold;color:white;">{len(high)}</div>
            <div style="color:#ffe0b2;font-size:12px;">🟠 HIGH<br>≤ 7 days</div>
          </div>
          <div style="background:linear-gradient(135deg,#ffd32a,#f39c12); padding:18px;border-radius:12px;text-align:center;">
            <div style="font-size:32px;font-weight:bold;color:white;">{len(medium)}</div>
            <div style="color:#fff9c4;font-size:12px;">🟡 MEDIUM<br>≤ 14 days</div>
          </div>
          <div style="background:linear-gradient(135deg,#2ed573,#27ae60); padding:18px;border-radius:12px;text-align:center;">
            <div style="font-size:32px;font-weight:bold;color:white;">{len(safe)}</div>
            <div style="color:#c8e6c9;font-size:12px;">🟢 SAFE<br>> 14 days</div>
          </div>
        </div>"""

        plan = "📋 ACTION PLAN\n" + "═"*50 + "\n\n"
        if critical:
            plan += "🔴 TURANT ORDER KARO (AAJ!):\n"
            for r in critical:
                cost = r['order_qty'] * store.inventory[r['item']]['price']
                plan += f"  • {r['item']:<18} Stock:{r['stock']:>4} | Order:{r['order_qty']:>4} | ₹{cost:,}\n"
        if high:
            plan += "\n🟠 IS HAFTE ORDER KARO:\n"
            for r in high:
                cost = r['order_qty'] * store.inventory[r['item']]['price']
                plan += f"  • {r['item']:<18} Stock:{r['stock']:>4} | Order:{r['order_qty']:>4} | ₹{cost:,}\n"
        if medium:
            plan += "\n🟡 IS MAHINE ORDER KARO:\n"
            for r in medium:
                plan += f"  • {r['item']:<18} Stock:{r['stock']:>4} | Days Left: {r['days_left']}\n"

        urgent_cost = sum(r['order_qty']*store.inventory[r['item']]['price'] for r in critical+high)
        plan += f"\n{'═'*50}\n💰 Total Urgent Cost: ₹{urgent_cost:,}\n"
        plan += f"📅 {datetime.now().strftime('%d %b %Y, %H:%M')}"

        table = pd.DataFrame([{
            'Item': r['item'], 'Stock': r['stock'], 'Daily Sale': r['avg_sale'],
            'Days Left': r['days_left'], 'Stock Out': r['stockout'],
            'Risk': r['risk'], 'Order Qty': r['order_qty'],
        } for r in results])

        chart = make_dashboard(results, store)
        return html, plan, table, chart
    except Exception as e:
        import traceback
        err_html = f"<div style='color:#ff4757; padding:20px'>❌ Prediction Error: {str(e)}</div>"
        return err_html, "Error occurred", pd.DataFrame(), None

def item_forecast(item_name):
    if not item_name:
        return None, "Item select karo!"
    try:
        pred_engine.train()
        stockout, preds = pred_engine.predict_item(item_name)
        props = store.inventory.get(item_name, {})
        dl = (props.get('stock',0)/props.get('avg_daily_sale',1) if props.get('avg_daily_sale',0)>0 else 999)
        risk = ('🔴 CRITICAL' if dl<=3 else '🟠 HIGH' if dl<=7 else '🟡 MEDIUM' if dl<=14 else '🟢 SAFE')
        so_txt = stockout.strftime('%d %b %Y') if stockout else '>30 days'
        info = f"""📦 {item_name}\n{'═'*38}\nCurrent Stock    : {props.get('stock',0)} units\nAvg Daily Sale   : {props.get('avg_daily_sale',0)} units/day\nReorder Point    : {props.get('reorder',0)} units\nPrice            : ₹{props.get('price',0)}\nDays Left        : {dl:.1f}\nRisk Level       : {risk}\nStock Out Date   : {so_txt}\nSuggested Order  : {int(props.get('avg_daily_sale',0)*30)} units\nEst. Order Cost  : ₹{int(props.get('avg_daily_sale',0)*30*props.get('price',0)):,}\n{'═'*38}"""
        chart = make_item_chart(item_name, preds, store)
        return chart, info
    except Exception as e:
        return None, f"❌ Error: {str(e)}"

# ── WhatsApp Auto-Draft Link Generator (FOR FINTERNSHIP TASK) ──
def generate_wa_link():
    try:
        results = pred_engine.all_predictions()
        urgent = [r for r in results if '🔴' in r['risk'] or '🟠' in r['risk']]
        if not urgent:
            return "<p style='color:#2ed573; margin-top:8px;'>✅ Abhi koi urgent stock mangwane ki zaroorat nahi hai.</p>"

        msg = "Bhaiya, apni dukan ke liye ye saaman jaldi bhej dena: %0A%0A"
        for r in urgent:
            msg += f"📦 *{r['item']}* - {r['order_qty']} pcs%0A"

        link = f"https://api.whatsapp.com/send?text={msg}"
        return f"<a href='{link}' target='_blank' style='display:inline-block; margin-top:10px; padding:10px 20px; background:linear-gradient(135deg, #25D366, #128C7E); color:white; border-radius:8px; text-decoration:none; font-weight:bold; font-size:14px; box-shadow:0 4px 10px rgba(37,211,102,0.3);'>📱 Supplier ko WhatsApp Bhejo</a>"
    except Exception as e:
        return f"<p style='color:red'>WhatsApp link error: {e}</p>"

# ── Inventory Handlers (Same as before) ───────────────────
def manual_sale(item, qty, note):
    try:
        qty = int(qty)
        ok  = store.add_sale(item, qty)
        msg = (f"✅ Sale recorded!\n{item}: -{qty} units\nNew Stock: {store.inventory[item]['stock']}\nNote: {note}" if ok else f"❌ {item} nahi mila!")
    except Exception as e: msg = f"❌ Error: {e}"
    return msg, store.get_inventory_df()

def manual_restock(item, qty, note):
    try:
        qty = int(qty)
        ok  = store.add_stock(item, qty)
        msg = (f"✅ Stock added!\n{item}: +{qty} units\nNew Stock: {store.inventory[item]['stock']}\nNote: {note}" if ok else f"❌ {item} nahi mila!")
    except Exception as e: msg = f"❌ Error: {e}"
    return msg, store.get_inventory_df()

def add_new_item(name, stock, avg, reorder, price):
    if not name.strip(): return "❌ Name daalo!", store.get_inventory_df()
    try:
        store.add_new_item(name.strip(), int(stock), float(avg), int(reorder), float(price))
        return f"✅ '{name}' add ho gaya!", store.get_inventory_df()
    except Exception as e: return f"❌ Error: {e}", store.get_inventory_df()

def qm_handler(item, qty, note, action):
    if "Sale" in action: return manual_sale(item, qty, note)
    return manual_restock(item, qty, note)

print("✅ Sab handlers & WhatsApp feature ready!")
print("\n✅ Cell 4 done! Ab Cell 5 chalao.")

✅ Sab handlers & WhatsApp feature ready!

✅ Cell 4 done! Ab Cell 5 chalao.


In [27]:
# ============================================================
# CELL 5 - GRADIO UI + LAUNCH (FIXED DATA FLOW)
# ============================================================
import gradio as gr

css = """
* { font-family:'Segoe UI',Arial,sans-serif !important; }
.gradio-container { background:linear-gradient(135deg,#0f0f23 0%,#1a1a3e 100%) !important; }
.main-header { background:linear-gradient(135deg,#667eea,#764ba2); padding:20px; border-radius:15px; text-align:center; margin-bottom:16px; box-shadow:0 8px 32px rgba(102,126,234,.3); }
.sec { color:#a0a0ff; font-size:14px; font-weight:bold; margin:10px 0 6px; padding:8px 12px; background:#1a1a3e; border-left:3px solid #667eea; border-radius:0 8px 8px 0; }
label { color:#c0c0e0 !important; font-size:13px !important; }
.sbox textarea { background:#0f1529 !important; color:#c0d0ff !important; font-family:'Courier New',monospace !important; font-size:12px !important; border:1px solid #3a3a5a !important; }
"""

with gr.Blocks(css=css, title="🏪 Stationary Shop AI") as app:
    gr.HTML("""
    <div class="main-header">
      <h1 style="color:white;margin:0;font-size:26px;">🏪 Stationary Shop AI System</h1>
      <p style="color:#d0d0ff;margin:6px 0 0;font-size:13px;">Gemini Vision OCR  •  Stock Prediction  •  WhatsApp Automation</p>
    </div>""")

    with gr.Tabs():
        # ════════ TAB 1 – BILL SCANNER ════════
        with gr.Tab("📸 Bill Scanner"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML('<div class="sec">1️⃣ Input (Photo/Text)</div>')
                    with gr.Tabs():
                        with gr.Tab("🖼️ Image Upload"):
                            img_in = gr.Image(type="pil", label="Bill / Invoice Photo", height=250)
                            img_btn = gr.Button("🔍 Gemini se Scan Karo", variant="primary")
                        with gr.Tab("⌨️ Text Input"):
                            txt_in = gr.Textbox(label="Bill text yahan type karo", lines=5)
                            txt_btn = gr.Button("🔍 Text Scan Karo", variant="primary")

                    scan_status = gr.Textbox(label="📋 Scan Status", lines=3, interactive=False, elem_classes=["sbox"])

                with gr.Column(scale=2):
                    gr.HTML('<div class="sec">2️⃣ Confirm & Inventory Update</div>')

                    # 💡 Data directly yahan aayega ab!
                    confirm_tbl = gr.Dataframe(
                        headers=['Item','Quantity','Unit','Brand','Confidence','Original Text'],
                        datatype=['str','number','str','str','str','str'],
                        label="Final Items (edit karke confirm karo)",
                        interactive=True,
                        row_count=(3, "dynamic")
                    )

                    with gr.Row():
                        action_dd = gr.Dropdown(choices=["Sale (Stock Ghata Do)", "Restock (Stock Badha Do)"], value="Sale (Stock Ghata Do)", label="Action Type", scale=2)
                        confirm_btn = gr.Button("✅ Confirm & Update", variant="primary", scale=1)

                    confirm_st = gr.Textbox(label="Update Status", lines=2, interactive=False, elem_classes=["sbox"])

            # Quick manual entry
            gr.HTML('<div class="sec">⚡ Quick Manual Entry</div>')
            with gr.Row():
                qm_item = gr.Dropdown(choices=list(store.inventory.keys()), label="Item", scale=2)
                qm_qty  = gr.Number(label="Qty", value=1, minimum=1, scale=1)
                qm_note = gr.Textbox(label="Note", scale=2)
                qm_act  = gr.Dropdown(choices=["Sale (Stock Ghata Do)", "Restock (Stock Badha Do)"], value="Sale (Stock Ghata Do)", label="Action", scale=1)
                qm_btn  = gr.Button("➕ Add", scale=1)

            qm_st = gr.Textbox(label="Status", lines=1, interactive=False, elem_classes=["sbox"])

        # ════════ TAB 2 – PREDICTIONS ════════
        with gr.Tab("📊 Stock Predictions"):
            gr.HTML('<div class="sec">🤖 AI Stock Out Prediction & Auto-Order</div>')

            with gr.Row():
                pred_btn = gr.Button("🚀 Prediction Chalao!", variant="primary", size="lg", scale=2)
                wa_btn = gr.Button("💬 WhatsApp Draft Banao", variant="secondary", size="lg", scale=1)

            pred_html = gr.HTML("<p style='color:#8080aa;text-align:center;padding:10px;'>↑ Button dabao prediction ke liye</p>")
            wa_html = gr.HTML() # Yahan WhatsApp link generate hoga

            with gr.Row():
                with gr.Column(scale=2):
                    pred_chart = gr.Image(label="📈 Dashboard", type="pil", height=420)
                with gr.Column(scale=1):
                    pred_plan = gr.Textbox(label="📋 Action Plan", lines=22, interactive=False, elem_classes=["sbox"])

            pred_table = gr.Dataframe(label="📦 All Items Prediction", interactive=False)

        # ════════ TAB 3 – ITEM DETAIL ════════
        with gr.Tab("🔍 Item Detail"):
            gr.HTML('<div class="sec">🔍 Kisi Ek Item Ki Detail Forecast</div>')
            with gr.Row():
                item_dd = gr.Dropdown(choices=list(store.inventory.keys()), value=list(store.inventory.keys())[0], label="Item Select Karo", scale=3)
                item_btn = gr.Button("🔮 Forecast Dekho", variant="primary", scale=1)

            with gr.Row():
                with gr.Column(scale=2):
                    item_chart = gr.Image(label="📉 Forecast Chart", type="pil", height=320)
                with gr.Column(scale=1):
                    item_info = gr.Textbox(label="📊 Details", lines=14, interactive=False, elem_classes=["sbox"])

        # ════════ TAB 4 – INVENTORY ════════
        with gr.Tab("📦 Inventory"):
            gr.HTML('<div class="sec">📦 Current Stock Status</div>')
            ref_btn = gr.Button("🔄 Refresh", variant="secondary")
            inv_table = gr.Dataframe(value=store.get_inventory_df(), label="Inventory", interactive=False)

    # ════════ EVENT WIRING ════════
    # OCR to Confirm Table directly
    img_btn.click(fn=scan_image, inputs=[img_in], outputs=[scan_status, confirm_tbl])
    txt_btn.click(fn=scan_text, inputs=[txt_in], outputs=[scan_status, confirm_tbl])

    # Update logic
    confirm_btn.click(fn=confirm_update, inputs=[confirm_tbl, action_dd], outputs=[confirm_st, inv_table])
    qm_btn.click(fn=qm_handler, inputs=[qm_item, qm_qty, qm_note, qm_act], outputs=[qm_st, inv_table])

    # Prediction & WhatsApp
    pred_btn.click(fn=run_prediction, inputs=[], outputs=[pred_html, pred_plan, pred_table, pred_chart])
    wa_btn.click(fn=generate_wa_link, inputs=[], outputs=[wa_html])

    # Item Details
    item_btn.click(fn=item_forecast, inputs=[item_dd], outputs=[item_chart, item_info])

    # Refresh
    ref_btn.click(fn=store.get_inventory_df, inputs=[], outputs=[inv_table])

# ── Launch ──
print("🚀 App launch ho raha hai...")
app.launch(share=True, debug=False, quiet=True)

🚀 App launch ho raha hai...
* Running on public URL: https://21e87d52f027f45364.gradio.live
